# NB-07: Patchify, Unpatchify, and the Head Module

## Learning Objectives
- Trace `WanModel.patchify`: how `Conv3d(48->1536, kernel=(1,2,2), stride=(1,2,2))` converts video `(B,48,F,H,W)` into token sequences `(B, F*(H//2)*(W//2), 1536)`
- Understand the spatial-to-sequence mapping: one latent frame maps to `H//2 x W//2` tokens; temporal dimension is fully preserved (patch_size[0]=1)
- Trace `WanModel.unpatchify`: how einops rearrange reverses the token sequence back to video shape `(B,16,F,H,W)`
- See how `Head` applies 2-parameter adaLN to project from model dim to output latent channels

## Prerequisites
- **Prior notebooks:** NB-01 (modulate function -- Head uses adaLN conditioning), NB-06 (DiTBlock -- patchify produces the token sequence that flows through DiT blocks)
- **Assumed concepts:** Conv3d kernel/stride mechanics, einops rearrange axis labeling

## Concept Map
- `WanModel.patchify` -- converts the 48-channel input into the sequence that DiT blocks process (NB-06, NB-08)
- `WanModel.unpatchify` -- reconstructs output video from the Head's projected tokens (NB-08)
- `Head` -- final output projection applied after all DiT blocks, before unpatchify (NB-08)

## Spatial-to-Sequence Mapping (patch_size=(1,2,2))

```
Spatial-to-Sequence Mapping (patch_size=(1,2,2))
=================================================
Input:  [B, 48, F, H, W]   e.g. [1, 48, 4, 8, 8]

         Frame f=0         Frame f=1        ...  Frame f=3
        +----------+      +----------+          +----------+
        | H=8, W=8 |  ->  | H=8, W=8 |          | H=8, W=8 |
        +----------+      +----------+          +----------+
              | Conv3d(stride=1,2,2): halves H and W per frame
              v
        +----------+      +----------+          +----------+
        | h=4, w=4 |      | h=4, w=4 |          | h=4, w=4 |   -> 16 tokens per frame
        +----------+      +----------+          +----------+
              | rearrange: 'b c f h w -> b (f h w) c'
              v
        Token sequence: [B, F*h*w, dim] = [1, 64, 1536]
        Position:        [0..15]  = frame 0 tokens
                         [16..31] = frame 1 tokens
                         [32..47] = frame 2 tokens
                         [48..63] = frame 3 tokens

Note: patch_size[0]=1 means temporal resolution is FULLY PRESERVED.
      Each latent frame becomes its own set of h*w tokens.
```

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# NB-07 imports
from diffsynth.models.wan_video_dit import WanModel, Head
import torch
import torch.nn as nn
import math
from einops import rearrange

print("Setup complete.")

## 1. Patchify -- Video to Token Sequence

`WanModel.patchify` converts a 5D video tensor into a 2D token sequence using a two-step process:

1. **Conv3d projection** (`patch_embedding`): Applies a learned convolution with kernel `(1,2,2)` and stride `(1,2,2)`
   - Kernel `(1,2,2)` means: 1 frame at a time, 2x2 spatial patch
   - Stride `(1,2,2)` means: step 1 frame at a time (no temporal overlap), step 2 pixels spatially (halves H and W)
   - This maps `(B, 48, F, H, W) -> (B, 1536, F, H//2, W//2)`

2. **Rearrange** (`einops.rearrange`): Flattens the spatial-temporal grid into a flat sequence
   - `'b c f h w -> b (f h w) c'` flattens `(F, H//2, W//2)` into a sequence of `F * H//2 * W//2` tokens
   - Result: `(B, seq_len, dim)` -- the format DiT blocks expect

**Temporal preservation:** `patch_size[0]=1` means every input frame produces a full set of `(H//2 * W//2)` tokens. The temporal dimension is preserved at full resolution in token space.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 340-350

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 340-350
# patchify = Conv3d projection + rearrange to sequence

patch_size = (1, 2, 2)
in_dim, dim = 48, 1536

patch_embedding = nn.Conv3d(in_dim, dim, kernel_size=patch_size, stride=patch_size)
# weight shape: (1536, 48, 1, 2, 2) -- 1536 filters, each 48-channel x 1x2x2 kernel
# bias shape:   (1536,)
print(f"patch_embedding.weight shape: {patch_embedding.weight.shape}")
print(f"patch_embedding.bias shape:   {patch_embedding.bias.shape}")

B, C, F, H, W = 1, 48, 4, 8, 8
x = torch.randn(B, C, F, H, W)          # [1, 48, 4, 8, 8]  -- 48-channel video input
px = patch_embedding(x)                   # [1, 1536, 4, 4, 4] -- stride halves H and W
b, c, f, h, w = px.shape                 # b=1, c=1536, f=4, h=4, w=4

print(f"\nInput:  {x.shape}   -- [B, C=48, F, H, W]")
print(f"After Conv3d: {px.shape}  -- [B, dim, F, H//2, W//2]")

# Source: diffsynth/models/wan_video_dit.py, line 349
x_seq = rearrange(px, 'b c f h w -> b (f h w) c')  # [1, 64, 1536]
assert x_seq.shape == torch.Size([B, f * h * w, dim]), \
    f"Expected ({B}, {f*h*w}, {dim}), got {x_seq.shape}"
print(f"After rearrange: {x_seq.shape}  -- [B, F*h*w, dim] = [B, seq_len, dim]")
print(f"patchify: {x.shape} -> Conv3d -> {px.shape} -> rearrange -> {x_seq.shape}")
print(f"\nseq_len = F*h*w = {F}*{h}*{w} = {F*h*w}  (4 frames x 4x4 spatial tokens per frame)")

## 2. Unpatchify -- Token Sequence Back to Video

`WanModel.unpatchify` reverses the sequence-to-video conversion using a single `einops.rearrange` call.

The input to unpatchify is the Head's output: `(B, seq_len, out_dim * prod(patch_size))`. For the production config, `out_dim=16` and `prod((1,2,2))=4`, so the Head produces `(B, seq_len, 64)` -- 64 values per token.

The rearrange string `'b (f h w) (x y z c) -> b c (f x) (h y) (w z)'` has a specific axis decomposition:

| Label | Meaning | Value |
|-------|---------|-------|
| `f`, `h`, `w` | token grid dims (from patchify) | 4, 4, 4 |
| `x` | temporal patch size | 1 (patch_size[0]) |
| `y` | height patch size | 2 (patch_size[1]) |
| `z` | width patch size | 2 (patch_size[2]) |
| `c` | output channels | 16 (out_dim) |

**Pitfall:** Swapping `x`, `y`, `z`, `c` labels produces a wrong spatial layout. Always copy the exact rearrange string from `WanModel.unpatchify` (line 353-357).

**Note:** Patchify and unpatchify are NOT mathematical inverses -- patchify uses learned Conv3d weights while unpatchify is a deterministic reshape. But the SHAPE round-trip is exact: `(B,C,F,H,W) -> (B,S,dim) -> (B,out_dim,F,H,W)`.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 352-357

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 352-357
out_dim = 16
out_dim_full = out_dim * math.prod(patch_size)  # 16 * 1*2*2 = 64
print(f"out_dim_full = out_dim * prod(patch_size) = {out_dim} * {math.prod(patch_size)} = {out_dim_full}")

# Simulate Head output -- shape that Head produces: [B, seq, out_dim*prod(patch_size)]
head_output = torch.randn(B, f * h * w, out_dim_full)  # [1, 64, 64]
print(f"\nSimulated Head output: {head_output.shape}  -- [B, seq_len, out_dim_full]")

# Source: wan_video_dit.py lines 353-357 -- copy the EXACT rearrange string
x_video = rearrange(
    head_output, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
    f=f, h=h, w=w, x=patch_size[0], y=patch_size[1], z=patch_size[2]
)  # [1, 16, 4, 8, 8]

assert x_video.shape == torch.Size([B, out_dim, F, H, W]), \
    f"Expected ({B}, {out_dim}, {F}, {H}, {W}), got {x_video.shape}"
print(f"After unpatchify: {x_video.shape}  -- [B, out_dim, F, H, W]")
print(f"unpatchify: {head_output.shape} -> rearrange -> {x_video.shape}")

### Patchify/Unpatchify Round-Trip Verification

Patchify and unpatchify are not inverses in terms of **values** -- patchify applies a learned Conv3d that changes the content. But the **shape** round-trip is exact:

- Input: `(B, C=48, F, H, W)` -- 48-channel video latents
- After patchify: `(B, seq_len, dim=1536)` -- token sequence
- After Head + unpatchify: `(B, out_dim=16, F, H, W)` -- output video latents

The spatial dimensions `(F, H, W)` match exactly between input and output. The channel dimension changes (48 in, 16 out) because patchify is a learned projection while unpatchify is a fixed reshape of the Head's output.

In [ ]:
# Shape round-trip verification
assert x.shape[2:] == x_video.shape[2:], \
    f"Spatial dimensions must match: {x.shape[2:]} != {x_video.shape[2:]}"
print(f"Input spatial:  {x.shape[2:]}  (F={F}, H={H}, W={W})")
print(f"Output spatial: {x_video.shape[2:]}  (F={F}, H={H}, W={W})")
print(f"Channel change: {x.shape[1]} (in_dim=48) -> {x_video.shape[1]} (out_dim=16)")
print()
print(f"Full shape trace:")
print(f"  Input video:   {x.shape}      <- 48 = 16 noise + 16 ctrl + 16 ref latents")
print(f"  After patchify:{x_seq.shape}    <- token sequence for DiT blocks (NB-06)")
print(f"  Head output:   {head_output.shape}    <- 64 = out_dim(16) * prod(patch_size)(4)")
print(f"  After unpatch: {x_video.shape}  <- output latents (16-channel, same spatial dims)")

## 3. Head Module -- Output Projection with adaLN

The `Head` module is the final projection applied after all DiT blocks, before unpatchify. It:
1. Applies `LayerNorm` (no affine weights -- the adaLN provides the affine transformation)
2. Applies 2-parameter adaLN conditioning (`shift` + `scale`)
3. Projects from model dimension to output channels: `dim -> out_dim * prod(patch_size)` = `1536 -> 64`

**Important -- `t` vs `t_mod` distinction:**

The `Head` receives **`t`** -- the 1536-dim time embedding output of `WanModel.time_embedding`, BEFORE the 6-chunk projection. This is different from the **`t_mod`** shape `[B, 6, dim]` that `DiTBlock` receives.

| Module | Receives | Shape | Parameters |
|--------|----------|-------|----------|
| `DiTBlock` | `t_mod` (6-chunk) | `[B, 6, dim]` | 6 params: shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp |
| `Head` | `t` (raw embedding) | `[B, dim]` | 2 params: shift, scale (no gate) |

The `Head` applies its **own** 2-parameter adaLN via `self.modulation` (shape `[1, 2, dim]`), NOT the 6-parameter modulation from DiTBlock.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 254-270

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 254-270
dim, out_dim, patch_size = 1536, 16, (1, 2, 2)
head = Head(dim=dim, out_dim=out_dim, patch_size=patch_size, eps=1e-6)

print("Head sub-modules:")
print(f"  norm (LayerNorm, no affine):          {sum(p.numel() for p in head.norm.parameters()):,} params")
print(f"  head (Linear {dim}->{out_dim * math.prod(patch_size)}):  {sum(p.numel() for p in head.head.parameters()):,} params")
print(f"  modulation (nn.Parameter [1,2,{dim}]): {head.modulation.numel():,} params")
print(f"  Head total:                            {sum(p.numel() for p in head.parameters()):,} params")

B, S = 1, 64
x_seq = torch.randn(B, S, dim)       # [B, seq, dim]   -- output of final DiT block
t = torch.randn(B, dim)               # [B, dim]         -- time embedding (BEFORE time_projection)
# NOTE: Head receives 't' (output of time_embedding), NOT 't_mod' (the 6-chunk [B,6,dim] tensor)
# Source: diffsynth/models/wan_video_dit.py, line 414: x = self.head(x, t)
# Head applies its own 2-parameter adaLN (shift + scale only, no gate)

head.eval()
with torch.no_grad():
    out = head(x_seq, t)              # [B, S, out_dim * prod(patch_size)] = [1, 64, 64]

assert out.shape == torch.Size([B, S, out_dim * math.prod(patch_size)]), \
    f"Expected ({B}, {S}, {out_dim * math.prod(patch_size)}), got {out.shape}"
print(f"\nInput x_seq: {x_seq.shape}   -- [B, seq, dim]")
print(f"Input t:     {t.shape}        -- [B, dim]  (raw time embedding, NOT 6-chunk t_mod)")
print(f"Head output: {out.shape}     -- [B, seq, out_dim * prod(patch_size)]")
print(f"  = [B, {S}, {out_dim}*{math.prod(patch_size)}] = [B, {S}, {out_dim * math.prod(patch_size)}]")

### Head's Place in the WanModel Pipeline

The Head connects the DiT block output to the unpatchify operation:

```
patchify -> N x DiTBlock -> Head -> unpatchify
[B,48,F,H,W]  [B,64,1536]  [B,64,1536]  [B,64,64]  [B,16,F,H,W]
```

The Head output `[B, S, 64]` is exactly what unpatchify expects: `64 = out_dim * prod(patch_size) = 16 * 1 * 2 * 2`. Unpatchify decomposes those 64 values per token into `(patch_t=1, patch_h=2, patch_w=2, c=16)` and reassembles them into the output video spatial layout.

The modulation in `Head` provides a **final timestep-conditioned rescaling** of the normalized features before projection. This lets the model adjust the output magnitude and bias based on the diffusion timestep, which is useful because denoising targets look very different at different noise levels.

## Summary

### Key Takeaways
- **patchify**: `Conv3d(48->1536, kernel=(1,2,2), stride=(1,2,2))` followed by `rearrange 'b c f h w -> b (f h w) c'`. Temporal `patch_size[0]=1` means every latent frame becomes its own set of tokens -- temporal resolution is fully preserved in token space.
- **unpatchify**: The exact rearrange string `'b (f h w) (x y z c) -> b c (f x) (h y) (w z)'` -- always copy from `WanModel.unpatchify` (line 353) to avoid axis-order bugs. The shape round-trip `(B,C,F,H,W) -> (B,S,dim) -> (B,out_dim,F,H,W)` preserves spatial dimensions exactly.
- **Head**: Applies 2-parameter adaLN (shift + scale, **no gate**) using the raw time embedding `t` -- NOT the 6-chunk `t_mod` used by DiTBlocks. Projects `dim -> out_dim * prod(patch_size) = 1536 -> 64`, which unpatchify reconstructs to 16 output channels.

### Source References
| Symbol | Location |
|--------|----------|
| `WanModel.patchify` | `diffsynth/models/wan_video_dit.py`, line 340 |
| `WanModel.unpatchify` | `diffsynth/models/wan_video_dit.py`, line 352 |
| `Head.__init__` | `diffsynth/models/wan_video_dit.py`, line 255 |
| `Head.forward` | `diffsynth/models/wan_video_dit.py`, line 263 |

## Exercises

### Exercise 1 -- Change patch size to (1,4,4)
Change `patch_size = (1, 4, 4)` in the patchify cell. Rerun patchify with `B, C, F, H, W = 1, 48, 4, 8, 8`. What is the new token sequence shape? What is the new Conv3d weight shape? What is `out_dim_full` for the unpatchify? (Hint: `prod((1,4,4)) = 16`, so `out_dim_full = 16 * 16 = 256`.)

### Exercise 2 -- End-to-end Head + unpatchify round-trip
Run the Head forward pass with the `x_seq` from patchify, then feed the Head's output directly into unpatchify. Verify that the resulting `x_video` has spatial dimensions `(F=4, H=8, W=8)` matching the original input. Check `assert x_video.shape[2:] == x.shape[2:]`.

### Exercise 3 -- Compare Conv3d parameter count to a DiTBlock
From NB-06, a single DiTBlock has 51,163,904 parameters. The `patch_embedding` Conv3d has weight shape `(1536, 48, 1, 2, 2)` plus bias `(1536,)`. Calculate: `1536 * 48 * 1 * 2 * 2 + 1536 = ?`. What fraction of one DiTBlock is the patch embedding? What fraction of the full 30-block model (approximately 30 * 51M parameters)?